# Solasterid Process Visualizer 

This companion notebook reads the **same `pentamind_memory_v3/` folder** used by your Solasterid/Pentamind notebook and renders a representative animation of the system's evolution.

The visual metaphor:

- The central disc is **Speakerbot / the mouth hole**.
- Each main radial arm is a **committee**.
- When a committee acts, its member arms sprout from that committee arm, creating the fractal starfish effect.
- Speech bubbles show each participating arm's name and the first ~50 words of its committee quote.
- Speakerbot scenes show the complete final finding, paginated if needed.
- Mutation scenes show architecture changes before the next run begins.

The notebook is intentionally data-driven. It does **not** import your main notebook or change any Solasterid parameters. It only reads run artifacts like `architecture_before.json`, `transcript.json`, `evolution_proposal.json`, and `mutation_results.json`.

In [ ]:
# Optional install helpers. Safe to skip if these packages already exist.
# imageio-ffmpeg is used for MP4 output; if unavailable, the notebook falls back to GIF.
import sys, subprocess, importlib.util

import matplotlib

_PACKAGES = ["numpy", "matplotlib", "Pillow", "imageio", "imageio_ffmpeg"]
missing = [p for p in _PACKAGES if importlib.util.find_spec(p.replace("-", "_")) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("Visualization dependencies already available.")

In [ ]:
from __future__ import annotations

import json, math, os, re, textwrap, hashlib, itertools, warnings
from collections import defaultdict, Counter
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyBboxPatch, FancyArrowPatch
from matplotlib import patheffects as pe
from PIL import Image
import imageio.v2 as imageio
matplotlib.rcParams['text.parse_math'] = False
warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams["figure.dpi"] = 120
print("Imports ready.")

In [ ]:
# =========================
# Configuration knobs
# =========================

# Run this notebook from the same folder as your Solasterid notebook.
# It expects the memory folder created by your main notebook to be nearby.
STORAGE_ROOT = Path("pentamind_memory_v3")
RUNS_DIR = STORAGE_ROOT / "runs"
OUTPUT_DIR = Path("solasterid_viz_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Video settings.
FPS = 12
FIGSIZE = (16, 9)
FRAME_DPI = 300
BACKGROUND = "#090912"
TEXT_COLOR = "#F4F0FF"
MUTED_TEXT = "#B8B4CC"

# Scene timing.
RUN_OVERVIEW_SECONDS = 2.0
SPROUT_SECONDS = 0.75
COMMITTEE_HOLD_SECONDS = 3.0
SPEAKER_PAGE_SECONDS = 3.0
MUTATION_SECONDS = 3.0

# Control long videos.
MAX_RUNS = None                         # e.g. 5 for a quick sample
MAX_COMMITTEE_SCENES_PER_RUN = None     # e.g. 12 for a short cut
QUOTE_WORDS = 50
SPEAKER_WORDS_PER_PAGE = 95             # "entire finding" gets paginated across several frames

# Output. MP4 first, GIF fallback if your environment lacks ffmpeg.
VIDEO_PATH = OUTPUT_DIR / "solasterid_evolution.mp4"
GIF_FALLBACK_PATH = OUTPUT_DIR / "solasterid_evolution.gif"

# Role palette. Hex colors are deliberately vivid because the animation needs readable anatomy.
ROLE_COLORS = {
    "governance_safety": "#FF5C8A",
    "memory_continuity": "#4EA5FF",
    "mechanics_code": "#53D769",
    "evolution_planning": "#B77CFF",
    "domain_specialist": "#FFD166",
    "creative_synthesis": "#FF8BD1",
    "economy_protocol": "#2DD4BF",
    "literal_grounding": "#F97316",
    "speaker": "#F8F4E3",
    "unknown": "#A7A6BA",
}

print(f"Looking for runs in: {RUNS_DIR.resolve()}")
print(f"Video target: {VIDEO_PATH.resolve()}")

In [ ]:
# =========================
# IO helpers
# =========================

def load_json(path: Path, default=None):
    if not path or not path.exists():
        return default
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception as e:
        print(f"Could not load JSON: {path} -> {e}")
        return default


def safe_read_text(path: Path, default=""):
    if path.exists():
        try:
            return path.read_text(encoding="utf-8")
        except Exception:
            return default
    return default


def parse_dt(value):
    if not value:
        return None
    try:
        # Handles ISO strings ending in Z or with offsets.
        return datetime.fromisoformat(str(value).replace("Z", "+00:00"))
    except Exception:
        return None


def run_sort_key(run_dir: Path):
    prompt = load_json(run_dir / "prompt.json", {}) or {}
    transcript = load_json(run_dir / "transcript.json", {}) or load_json(run_dir / "transcript_so_far.json", {}) or {}
    dt = parse_dt(prompt.get("created_at") or transcript.get("created_at"))
    if dt:
        return dt.timestamp()
    return run_dir.stat().st_mtime


def discover_runs(runs_dir: Path = RUNS_DIR, max_runs: Optional[int] = MAX_RUNS):
    if not runs_dir.exists():
        print(f"No runs directory found: {runs_dir.resolve()}")
        return []
    runs = [p for p in runs_dir.iterdir() if p.is_dir()]
    runs = sorted(runs, key=run_sort_key)
    if max_runs is not None:
        runs = runs[-int(max_runs):]
    print(f"Found {len(runs)} run folders.")
    if runs:
        print("First:", runs[0].name)
        print("Latest:", runs[-1].name)
    return runs


def load_run_bundle(run_dir: Path):
    transcript = load_json(run_dir / "transcript.json", None)
    if transcript is None:
        transcript = load_json(run_dir / "transcript_so_far.json", {}) or {}
    arch_before = load_json(run_dir / "architecture_before.json", None)
    if arch_before is None:
        arch_before = load_json(run_dir / "architecture_snapshot.json", {}) or {}
    arch_after = load_json(run_dir / "architecture_after_mutation.json", None)
    if arch_after is None:
        arch_after = load_json(run_dir / "architecture_after.json", None)
    if arch_after is None:
        arch_after = arch_before
    return {
        "run_dir": run_dir,
        "run_id": (load_json(run_dir / "prompt.json", {}) or {}).get("run_id", run_dir.name),
        "prompt": load_json(run_dir / "prompt.json", {}) or {},
        "architecture_before": arch_before or {},
        "architecture_after": arch_after or arch_before or {},
        "transcript": transcript or {},
        "diagnostics": load_json(run_dir / "diagnostics.json", {}) or {},
        "evolution": load_json(run_dir / "evolution_proposal.json", {}) or {},
        "mutation_results": load_json(run_dir / "mutation_results.json", []) or [],
        "final_output": safe_read_text(run_dir / "final_output.txt", "") or (transcript or {}).get("final_output", ""),
    }

runs = discover_runs()

In [ ]:
# =========================
# Anatomy normalization
# =========================

def is_active_arm(arm: dict) -> bool:
    return isinstance(arm, dict) and not arm.get("retired", False) and arm.get("status", "active") != "retired"


def active_arms(state: dict) -> dict:
    arms = state.get("arms", {}) if isinstance(state.get("arms", {}), dict) else {}
    return {aid: a for aid, a in arms.items() if is_active_arm(a)}


def active_committees(state: dict) -> dict:
    committees = state.get("committees", {}) if isinstance(state.get("committees", {}), dict) else {}
    arms = active_arms(state)
    out = {}
    for cid, c in committees.items():
        if not isinstance(c, dict) or c.get("status", "active") == "retired":
            continue
        members = [m for m in c.get("members", []) if m in arms]
        leader = c.get("leader") if c.get("leader") in arms else (members[0] if members else None)
        if not leader and not members:
            continue
        c2 = dict(c)
        c2["members"] = members if members else ([leader] if leader else [])
        c2["leader"] = leader
        c2.setdefault("name", cid)
        c2.setdefault("layer", 0)
        c2.setdefault("parent", None)
        c2.setdefault("children", [])
        out[cid] = c2
    return out


def pseudo_committees_from_arms(state: dict) -> dict:
    """Old pre-committee runs still get visualized: each arm becomes a pseudo-committee."""
    out = {}
    for aid, arm in active_arms(state).items():
        out[f"pseudo_{aid}"] = {
            "name": arm.get("name", aid),
            "leader": aid,
            "members": [aid],
            "layer": 0,
            "parent": None,
            "children": [],
            "routing_hint": arm.get("lens", ""),
            "status": "active",
            "_pseudo": True,
        }
    return out


def committees_or_pseudo(state: dict) -> dict:
    c = active_committees(state)
    return c if c else pseudo_committees_from_arms(state)


def get_arm_name(state: dict, arm_id: str) -> str:
    return (state.get("arms", {}).get(arm_id, {}) or {}).get("name", arm_id)


def get_committee_name(state: dict, committee_id: str) -> str:
    return (committees_or_pseudo(state).get(committee_id, {}) or {}).get("name", committee_id)

In [ ]:
# =========================
# Role/color classification
# =========================

ROLE_PATTERNS = [
    ("governance_safety", r"schema|sentinel|assumption|audit|auditor|boundary|adversary|risk|safety|validator|absent-layer|semantics"),
    ("memory_continuity", r"memory|historian|replay|signal|continuity|curator|steward|remember"),
    ("mechanics_code", r"mechanic|computer|code|runtime|implementation|rollback|debug|software|cs\b|computer-science"),
    ("evolution_planning", r"growth|cycle|regime|benchmark|cartograph|planner|experiment|evolution|mutation|route|routing"),
    ("domain_specialist", r"math|science|medicine|medical|domain|specialist|physics|biology|chemistry"),
    ("creative_synthesis", r"dream|creative|synthesis|synthesist|metaphor|aesthetic"),
    ("economy_protocol", r"protocol|economist|incentive|cost|budget|governance"),
    ("literal_grounding", r"literal|ground|scope|definition|exact"),
]


def classify_text_role(*parts: str) -> str:
    blob = " ".join(str(p or "") for p in parts).lower()
    for role, pat in ROLE_PATTERNS:
        if re.search(pat, blob):
            return role
    return "unknown"


def committee_role(cid: str, committee: dict, state: dict) -> str:
    members = committee.get("members", []) or []
    member_text = []
    for aid in members:
        arm = state.get("arms", {}).get(aid, {}) or {}
        member_text.append(arm.get("name", ""))
        member_text.append(arm.get("lens", ""))
    return classify_text_role(cid, committee.get("name"), committee.get("routing_hint"), *member_text)


def arm_role(arm_id: str, state: dict) -> str:
    arm = state.get("arms", {}).get(arm_id, {}) or {}
    return classify_text_role(arm_id, arm.get("name"), arm.get("lens"))


def role_color(role: str) -> str:
    return ROLE_COLORS.get(role, ROLE_COLORS["unknown"])


def stable_jitter(key: str, scale=0.1):
    h = hashlib.md5(str(key).encode()).hexdigest()
    return ((int(h[:8], 16) / 0xFFFFFFFF) - 0.5) * 2 * scale

In [ ]:
# =========================
# Text extraction
# =========================

_WORD_RE = re.compile(r"\S+")


def clean_text(x: Any) -> str:
    if x is None:
        return ""
    if isinstance(x, (dict, list)):
        try:
            x = json.dumps(x, ensure_ascii=False)
        except Exception:
            x = str(x)
    s = str(x)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def first_words(text: Any, n: int = QUOTE_WORDS) -> str:
    s = clean_text(text)
    words = _WORD_RE.findall(s)
    if len(words) <= n:
        return s
    return " ".join(words[:n]) + " …"


def wrap_text(text: str, width: int = 34, max_lines: Optional[int] = None) -> str:
    s = clean_text(text)
    lines = textwrap.wrap(s, width=width, break_long_words=False, replace_whitespace=True)
    if max_lines is not None and len(lines) > max_lines:
        lines = lines[:max_lines]
        if lines:
            lines[-1] = lines[-1].rstrip(" .,") + " …"
    return "\n".join(lines)


def report_quote(report: dict, n_words: int = QUOTE_WORDS) -> str:
    cf = report.get("committee_finding")
    cf_value = cf.get("value") if isinstance(cf, dict) else ""
    for key_value in [
        cf_value,
        report.get("interpretation"),
        report.get("candidate_patch_or_phrase"),
        report.get("scratch"),
        report.get("candidate_final_answer"),
    ]:
        q = clean_text(key_value)
        if q:
            return first_words(q, n_words)
    return "(no quote captured)"


def split_words_into_pages(text: str, words_per_page: int = SPEAKER_WORDS_PER_PAGE) -> List[str]:
    words = _WORD_RE.findall(clean_text(text))
    if not words:
        return ["(Speakerbot produced no final text in this run.)"]
    pages = []
    for i in range(0, len(words), words_per_page):
        pages.append(" ".join(words[i:i + words_per_page]))
    return pages

In [ ]:
# =========================
# Committee-scene extraction
# =========================

@dataclass
class CommitteeScene:
    run_id: str
    step_index: int
    deliberation_round: int
    committee_id: str
    committee_name: str
    reports: List[dict]
    leader_event: Optional[dict] = None


def pseudo_committee_id_from_report(report: dict) -> str:
    aid = report.get("arm_id") or "unknown"
    return f"pseudo_{aid}"


def group_committee_scenes(bundle: dict, max_scenes: Optional[int] = MAX_COMMITTEE_SCENES_PER_RUN) -> List[CommitteeScene]:
    state = bundle["architecture_before"]
    committees = committees_or_pseudo(state)
    transcript = bundle.get("transcript", {}) or {}
    run_id = bundle.get("run_id") or bundle["run_dir"].name
    reports = transcript.get("arm_reports", []) or []
    events = transcript.get("committee_events", []) or []

    # Leader events keyed by chronological committee hop.
    event_by_key = {}
    for ev in events:
        if ev.get("type") == "committee_leader":
            key = (int(ev.get("step_index", 0) or 0), int(ev.get("deliberation_round", 0) or 0), ev.get("committee_id"))
            event_by_key[key] = ev

    grouped = defaultdict(list)
    order_seen = []
    for r in reports:
        step = int(r.get("step_index", 0) or 0)
        rnd = int(r.get("deliberation_round", 0) or 0)
        cid = r.get("committee_id")
        if not cid:
            cid = pseudo_committee_id_from_report(r)
        key = (step, rnd, cid)
        if key not in grouped:
            order_seen.append(key)
        grouped[key].append(r)

    # If no arm reports exist but committee events do, still render leader-only scenes.
    for ev in events:
        cid = ev.get("committee_id")
        if not cid:
            continue
        key = (int(ev.get("step_index", 0) or 0), int(ev.get("deliberation_round", 0) or 0), cid)
        if key not in grouped and key not in order_seen:
            order_seen.append(key)

    scenes = []
    for key in order_seen:
        step, rnd, cid = key
        c = committees.get(cid, {"name": cid})
        scenes.append(CommitteeScene(
            run_id=run_id,
            step_index=step,
            deliberation_round=rnd,
            committee_id=cid,
            committee_name=c.get("name", cid),
            reports=grouped.get(key, []),
            leader_event=event_by_key.get(key),
        ))

    if max_scenes is not None:
        scenes = scenes[:int(max_scenes)]
    return scenes


def summarize_bundle(bundle: dict):
    transcript = bundle.get("transcript", {}) or {}
    return {
        "run_id": bundle.get("run_id"),
        "committees": len(committees_or_pseudo(bundle.get("architecture_before", {}))),
        "arm_reports": len(transcript.get("arm_reports", []) or []),
        "committee_events": len(transcript.get("committee_events", []) or []),
        "speaker_decisions": len(transcript.get("speaker_decisions", []) or []),
        "mutation_results": len(bundle.get("mutation_results", []) or []),
        "final_words": len(_WORD_RE.findall(clean_text(bundle.get("final_output", "")))),
    }

if runs:
    sample_bundle = load_run_bundle(runs[-1])
    print(summarize_bundle(sample_bundle))
    print("Sample scenes:", len(group_committee_scenes(sample_bundle, max_scenes=5)))
else:
    print("No runs yet. Create runs in the main Solasterid notebook first, then re-run this cell.")

In [ ]:
# =========================
# Geometry + drawing primitives
# =========================

@dataclass
class CommitteeLayout:
    center: Tuple[float, float]
    angle: float
    end: Tuple[float, float]
    role: str
    color: str
    layer: int


def compute_committee_layout(state: dict, radius: float = 3.25) -> Dict[str, CommitteeLayout]:
    committees = committees_or_pseudo(state)
    if not committees:
        return {}

    # Sort by layer, then id, but keep stable spacing.
    items = sorted(committees.items(), key=lambda kv: (int((kv[1] or {}).get("layer", 0) or 0), kv[0]))
    n = len(items)
    # Put the first committee at the top, then clockwise. A little offset prevents perfect symmetry from hiding labels.
    start = math.pi / 2 + (math.pi / max(n, 3)) * 0.15
    layout = {}
    for idx, (cid, c) in enumerate(items):
        angle = start - idx * 2 * math.pi / max(n, 1)
        layer = int(c.get("layer", 0) or 0)
        # Subtle layer radius offset: nested/higher-layer committees sit a touch farther out.
        r = radius + min(layer, 3) * 0.22
        x, y = r * math.cos(angle), r * math.sin(angle)
        role = committee_role(cid, c, state)
        layout[cid] = CommitteeLayout(center=(0, 0), angle=angle, end=(x, y), role=role, color=role_color(role), layer=layer)
    return layout


def draw_glow_line(ax, x0, y0, x1, y1, color, lw=28, alpha=0.85, z=2, glow=True):
    line, = ax.plot([x0, x1], [y0, y1], color=color, lw=lw, alpha=alpha, solid_capstyle="round", zorder=z)
    if glow:
        line.set_path_effects([
            pe.Stroke(linewidth=lw + 10, foreground=color, alpha=0.12),
            pe.Stroke(linewidth=lw + 4, foreground=color, alpha=0.20),
            pe.Normal(),
        ])
    return line


def draw_label(ax, x, y, text, color=TEXT_COLOR, size=9, ha="center", va="center", box=True, alpha=0.95, z=10):
    bbox = None
    if box:
        bbox = dict(boxstyle="round,pad=0.34,rounding_size=0.2", fc="#161527", ec=color, lw=1.0, alpha=alpha)
    return ax.text(x, y, text, color=color, fontsize=size, ha=ha, va=va, bbox=bbox, zorder=z)


def draw_speech_bubble(ax, x, y, text, edge_color, box_width=36, max_lines=7, size=7.2, ha=None, z=20):
    if ha is None:
        ha = "left" if x >= 0 else "right"
    wrapped = wrap_text(text, width=box_width, max_lines=max_lines)
    bbox = dict(boxstyle="round,pad=0.55,rounding_size=0.22", fc="#F8F5FF", ec=edge_color, lw=1.4, alpha=0.94)
    txt = ax.text(x, y, wrapped, color="#1B1733", fontsize=size, ha=ha, va="center", bbox=bbox, zorder=z)
    txt.set_path_effects([pe.withStroke(linewidth=2.5, foreground="white", alpha=0.25)])
    return txt


def draw_role_legend(ax):
    roles = [
        ("literal_grounding", "literal"),
        ("governance_safety", "safety/audit"),
        ("mechanics_code", "mechanics/code"),
        ("memory_continuity", "memory"),
        ("evolution_planning", "planning/evolution"),
        ("domain_specialist", "domain"),
        ("creative_synthesis", "creative"),
        ("economy_protocol", "protocol/econ"),
    ]
    x0, y0 = -6.9, -3.65
    for i, (role, label) in enumerate(roles):
        x = x0 + (i % 4) * 2.5
        y = y0 - (i // 4) * 0.45
        ax.scatter([x], [y], s=60, color=role_color(role), zorder=30)
        ax.text(x + 0.16, y, label, color=MUTED_TEXT, fontsize=7.5, ha="left", va="center", zorder=30)


def setup_axis(title=""):
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=FRAME_DPI)
    fig.patch.set_facecolor(BACKGROUND)
    ax.set_facecolor(BACKGROUND)
    ax.set_xlim(-7.2, 7.2)
    ax.set_ylim(-4.2, 4.2)
    ax.set_aspect("equal")
    ax.axis("off")
    if title:
        ax.text(-6.95, 3.95, title, color=TEXT_COLOR, fontsize=13, weight="bold", ha="left", va="top")
    return fig, ax

In [ ]:
# =========================
# Starfish renderer
# =========================

def draw_starfish_anatomy(
    ax,
    state: dict,
    selected_committee: Optional[str] = None,
    reports: Optional[List[dict]] = None,
    sprout_progress: float = 1.0,
    subtitle: str = "",
    show_all_labels: bool = True,
    mutation_highlights: Optional[dict] = None,
):
    committees = committees_or_pseudo(state)
    arms = active_arms(state)
    layout = compute_committee_layout(state)
    mutation_highlights = mutation_highlights or {}

    # faint parent-child or layer relationship threads
    for cid, c in committees.items():
        parent = c.get("parent")
        if parent in layout and cid in layout:
            x0, y0 = layout[parent].end
            x1, y1 = layout[cid].end
            ax.plot([x0, x1], [y0, y1], color="#FFFFFF", lw=1, ls="--", alpha=0.18, zorder=1)

    # main committee arms
    for cid, c in committees.items():
        if cid not in layout:
            continue
        L = layout[cid]
        x1, y1 = L.end
        selected = (cid == selected_committee)
        alpha = 0.95 if selected or selected_committee is None else 0.45
        lw = 34 if selected else 24
        color = L.color
        draw_glow_line(ax, 0, 0, x1, y1, color, lw=lw, alpha=alpha, z=3 if selected else 2, glow=True)
        ax.scatter([x1], [y1], s=400 if selected else 210, color=color, alpha=alpha, zorder=4, edgecolor="#FFF7", linewidth=1.0)

        # Mutation glow around changed committees.
        if cid in mutation_highlights.get("added_committees", set()):
            ax.scatter([x1], [y1], s=850, facecolor="none", edgecolor="#A7FF83", linewidth=2.5, alpha=0.85, zorder=5)
        if cid in mutation_highlights.get("changed_committees", set()):
            ax.scatter([x1], [y1], s=700, facecolor="none", edgecolor="#FFE66D", linewidth=2.0, alpha=0.8, zorder=5)

        if show_all_labels or selected:
            label = c.get("name", cid)
            if len(label) > 26:
                label = label[:25] + "…"
            lx = x1 + 0.34 * math.cos(L.angle)
            ly = y1 + 0.34 * math.sin(L.angle)
            draw_label(ax, lx, ly, label, color=TEXT_COLOR, size=7.8 if len(committees) > 10 else 8.8, box=True, alpha=0.72)

    # central speaker mouth hole
    ax.add_patch(Circle((0, 0), 0.62, facecolor="#08070D", edgecolor="#F8F4E3", linewidth=2.0, alpha=0.96, zorder=8))
    ax.add_patch(Circle((0, 0), 0.30, facecolor="#000000", edgecolor="#7C6CF2", linewidth=1.4, alpha=0.98, zorder=9))
    ax.text(0, -0.02, "Speaker\nmouth", color="#F8F4E3", fontsize=8, ha="center", va="center", zorder=10)

    if subtitle:
        ax.text(0, 3.85, subtitle, color=MUTED_TEXT, fontsize=9, ha="center", va="top", zorder=25)

    # selected committee member sprouts + bubbles
    if selected_committee and selected_committee in committees and selected_committee in layout:
        c = committees[selected_committee]
        L = layout[selected_committee]
        members = [m for m in c.get("members", []) if m in arms]
        # Include report arms even if the committee metadata changed since then.
        for r in reports or []:
            aid = r.get("arm_id")
            if aid and aid in arms and aid not in members:
                members.append(aid)

        # Map report per member, keep all if multiple. Prefer the last report in this committee scene.
        by_arm = defaultdict(list)
        for r in reports or []:
            by_arm[r.get("arm_id")].append(r)

        base_x = 0.55 * L.end[0]
        base_y = 0.55 * L.end[1]
        fan = np.linspace(-0.78, 0.78, max(len(members), 1)) if len(members) > 1 else [0]
        for j, aid in enumerate(members):
            arm = arms.get(aid, {})
            theta = L.angle + float(fan[j])
            length = (1.25 + 0.20 * ((j % 3) - 1)) * max(0.0, min(1.0, sprout_progress))
            start_x = base_x + 0.35 * math.cos(L.angle)
            start_y = base_y + 0.35 * math.sin(L.angle)
            end_x = start_x + length * math.cos(theta)
            end_y = start_y + length * math.sin(theta)
            acolor = role_color(arm_role(aid, state))
            draw_glow_line(ax, start_x, start_y, end_x, end_y, acolor, lw=10, alpha=0.88, z=12, glow=True)
            ax.scatter([end_x], [end_y], s=95, color=acolor, edgecolor="#FFFFFFAA", linewidth=0.8, zorder=13)
            name = arm.get("name", aid)
            if sprout_progress > 0.72:
                report = by_arm.get(aid, [{}])[-1]
                quote = report_quote(report, QUOTE_WORDS) if report else "(committee member active)"
                bubble = f"{name}: {quote}"
                # push bubbles outward and stagger vertically to reduce overlap
                bx = end_x + 0.55 * math.cos(theta)
                by = end_y + 0.55 * math.sin(theta) + stable_jitter(f"{selected_committee}-{aid}", 0.18)
                draw_speech_bubble(ax, bx, by, bubble, acolor, box_width=38, max_lines=7, size=6.6)

    return layout


def render_committee_scene(bundle: dict, scene: CommitteeScene, sprout_progress: float = 1.0):
    title = f"Solasterid run {bundle['run_id']}"
    subtitle = f"Step {scene.step_index} / round {scene.deliberation_round} · committee: {scene.committee_name}"
    fig, ax = setup_axis(title)
    draw_starfish_anatomy(
        ax,
        bundle["architecture_before"],
        selected_committee=scene.committee_id,
        reports=scene.reports,
        sprout_progress=sprout_progress,
        subtitle=subtitle,
    )
    draw_role_legend(ax)
    return fig


def render_overview_scene(bundle: dict):
    state = bundle["architecture_before"]
    prompt_text = clean_text((bundle.get("prompt", {}) or {}).get("user_prompt", ""))
    prompt_short = first_words(prompt_text, 45) if prompt_text else "(no prompt text captured)"
    summary = summarize_bundle(bundle)
    title = f"Solasterid run {bundle['run_id']} · anatomy overview"
    fig, ax = setup_axis(title)
    subtitle = f"{summary['committees']} committees · {summary['arm_reports']} reports · {summary['speaker_decisions']} speaker decisions"
    draw_starfish_anatomy(ax, state, subtitle=subtitle)
    panel = (
        f"Prompt:\n{wrap_text(prompt_short, 64, max_lines=5)}\n\n"
        f"Run folder: {bundle['run_dir'].name}"
    )
    draw_speech_bubble(ax, -6.65, 2.05, panel, "#8E85FF", box_width=58, max_lines=9, size=8.0, ha="left")
    draw_role_legend(ax)
    return fig


def render_speaker_scene(bundle: dict, page_text: str, page_idx: int, page_count: int):
    title = f"Speakerbot mouth hole · run {bundle['run_id']} · page {page_idx + 1}/{page_count}"
    fig, ax = setup_axis(title)
    draw_starfish_anatomy(ax, bundle["architecture_before"], show_all_labels=False)
    # Big central output box.
    box = FancyBboxPatch(
        (-4.75, -2.50), 9.50, 5.00,
        boxstyle="round,pad=0.03,rounding_size=0.22",
        facecolor="#F8F5FF", edgecolor="#F8F4E3", linewidth=2.0, alpha=0.96, zorder=30
    )
    ax.add_patch(box)
    ax.text(0, 2.20, "Speakerbot final finding", color="#1B1733", fontsize=13, weight="bold", ha="center", va="top", zorder=31)
    ax.text(0, 1.86, wrap_text(page_text, 92, max_lines=20), color="#1B1733", fontsize=9.2, ha="center", va="top", zorder=31)
    return fig


def mutation_summary_text(bundle: dict, max_items: int = 10) -> str:
    muts = bundle.get("mutation_results", []) or []
    evo = bundle.get("evolution", {}) or {}
    lines = []
    if muts:
        for m in muts[:max_items]:
            if isinstance(m, dict):
                typ = m.get("type") or m.get("mutation_type") or m.get("action") or "mutation"
                applied = m.get("applied")
                reason = m.get("reason") or m.get("notes") or m.get("target") or ""
                lines.append(f"• {typ} · {'applied' if applied else 'not applied'} · {reason}")
            else:
                lines.append(f"• {clean_text(m)}")
    else:
        proposed = evo.get("mutations") or evo.get("proposed_mutations") or []
        for m in proposed[:max_items]:
            if isinstance(m, dict):
                lines.append(f"• proposed {m.get('type') or m.get('mutation_type') or m.get('action') or 'mutation'}: {m.get('reason') or m.get('rationale') or m.get('target') or ''}")
            else:
                lines.append(f"• proposed {clean_text(m)}")
    if not lines:
        lines = ["• No mutation artifact found for this run."]
    return "\n".join(lines)


def diff_architecture(before: dict, after: dict) -> dict:
    b_c = committees_or_pseudo(before)
    a_c = committees_or_pseudo(after)
    added_committees = set(a_c) - set(b_c)
    changed_committees = set()
    for cid in set(a_c) & set(b_c):
        if (a_c[cid].get("leader") != b_c[cid].get("leader") or
            set(a_c[cid].get("members", [])) != set(b_c[cid].get("members", [])) or
            a_c[cid].get("parent") != b_c[cid].get("parent")):
            changed_committees.add(cid)
    b_a = active_arms(before)
    a_a = active_arms(after)
    return {
        "added_committees": added_committees,
        "changed_committees": changed_committees,
        "added_arms": set(a_a) - set(b_a),
        "retired_arms": set(b_a) - set(a_a),
    }


def render_mutation_scene(bundle: dict):
    before = bundle["architecture_before"]
    after = bundle.get("architecture_after") or before
    highlights = diff_architecture(before, after)
    title = f"Mutation tide · run {bundle['run_id']}"
    fig, ax = setup_axis(title)
    # Show after-anatomy so the next scene begins where the mutation landed.
    draw_starfish_anatomy(ax, after, subtitle="Architecture after mutation application", mutation_highlights=highlights)
    txt = mutation_summary_text(bundle)
    draw_speech_bubble(ax, -6.65, 2.45, "Mutation log:\n" + txt, "#A7FF83", box_width=58, max_lines=14, size=7.6, ha="left")
    draw_role_legend(ax)
    return fig

In [ ]:
# =========================
# Frame conversion + video writer
# =========================

def fig_to_pil(fig) -> Image.Image:
    fig.canvas.draw()
    w, h = fig.canvas.get_width_height()
    buf = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(h, w, 4)
    img = Image.fromarray(buf, mode="RGBA").convert("RGB")
    plt.close(fig)
    return img


class VideoWriter:
    def __init__(self, output_path: Path = VIDEO_PATH, fps: int = FPS, gif_fallback: Path = GIF_FALLBACK_PATH):
        self.output_path = Path(output_path)
        self.gif_fallback = Path(gif_fallback)
        self.fps = int(fps)
        self.writer = None
        self.frames_for_gif = []
        self.mode = "mp4"

    def __enter__(self):
        self.output_path.parent.mkdir(parents=True, exist_ok=True)
        try:
            self.writer = imageio.get_writer(str(self.output_path), fps=self.fps, codec="libx264", quality=8)
            self.mode = "mp4"
            print(f"Writing MP4: {self.output_path}")
        except Exception as e:
            print("MP4 writer unavailable; falling back to GIF. Reason:", e)
            self.writer = None
            self.mode = "gif"
        return self

    def add_image(self, img: Image.Image, seconds: float = 1.0):
        repeat = max(1, int(round(seconds * self.fps)))
        arr = np.asarray(img)
        if self.writer is not None:
            for _ in range(repeat):
                self.writer.append_data(arr)
        else:
            # Avoid huge GIFs: sample at ~6 fps if MP4 unavailable.
            gif_repeat = max(1, int(round(seconds * 6)))
            for _ in range(gif_repeat):
                self.frames_for_gif.append(img.copy())

    def __exit__(self, exc_type, exc, tb):
        if self.writer is not None:
            self.writer.close()
            print(f"Saved video: {self.output_path.resolve()}")
        elif self.frames_for_gif:
            self.gif_fallback.parent.mkdir(parents=True, exist_ok=True)
            self.frames_for_gif[0].save(
                self.gif_fallback,
                save_all=True,
                append_images=self.frames_for_gif[1:],
                duration=int(1000 / 6),
                loop=0,
            )
            print(f"Saved GIF fallback: {self.gif_fallback.resolve()}")
        return False


def add_fig(writer: VideoWriter, fig, seconds: float):
    writer.add_image(fig_to_pil(fig), seconds=seconds)

In [ ]:
# =========================
# Main render pipeline
# =========================

def render_solasterid_video(
    runs_dir: Path = RUNS_DIR,
    output_path: Path = VIDEO_PATH,
    max_runs: Optional[int] = MAX_RUNS,
    max_committee_scenes_per_run: Optional[int] = MAX_COMMITTEE_SCENES_PER_RUN,
    fps: int = FPS,
):
    run_dirs = discover_runs(runs_dir, max_runs=max_runs)
    if not run_dirs:
        raise FileNotFoundError(f"No run folders found in {runs_dir.resolve()}")

    with VideoWriter(output_path=output_path, fps=fps) as writer:
        for idx, run_dir in enumerate(run_dirs, start=1):
            bundle = load_run_bundle(run_dir)
            print(f"[{idx}/{len(run_dirs)}] Rendering {bundle['run_id']} ...")

            # 1) Anatomy overview.
            add_fig(writer, render_overview_scene(bundle), seconds=RUN_OVERVIEW_SECONDS)

            # 2) Committee-by-committee fractal scenes.
            scenes = group_committee_scenes(bundle, max_scenes=max_committee_scenes_per_run)
            for scene in scenes:
                # Short sprout animation.
                sprout_frame_count = max(1, int(SPROUT_SECONDS * fps))
                for k in range(sprout_frame_count):
                    p = (k + 1) / sprout_frame_count
                    fig = render_committee_scene(bundle, scene, sprout_progress=p)
                    add_fig(writer, fig, seconds=1 / fps)
                # Hold with all quotes visible.
                fig = render_committee_scene(bundle, scene, sprout_progress=1.0)
                add_fig(writer, fig, seconds=COMMITTEE_HOLD_SECONDS)

            # 3) Speakerbot mouth-hole scene with complete final output, paginated.
            final_text = bundle.get("final_output") or (bundle.get("transcript", {}) or {}).get("final_output", "")
            pages = split_words_into_pages(final_text, SPEAKER_WORDS_PER_PAGE)
            for pi, page in enumerate(pages):
                fig = render_speaker_scene(bundle, page, pi, len(pages))
                add_fig(writer, fig, seconds=SPEAKER_PAGE_SECONDS)

            # 4) Mutation scene, if any mutation/evolution artifact exists.
            if bundle.get("mutation_results") or bundle.get("evolution"):
                add_fig(writer, render_mutation_scene(bundle), seconds=MUTATION_SECONDS)

    return output_path

print("Render pipeline ready: call render_solasterid_video().")

In [ ]:
# =========================
# Preview latest run before making the full video
# =========================

if runs:
    latest = load_run_bundle(runs[-1])
    scenes = group_committee_scenes(latest, max_scenes=1)
    if scenes:
        fig = render_committee_scene(latest, scenes[0], sprout_progress=1.0)
    else:
        fig = render_overview_scene(latest)
    display(fig)
else:
    print("No runs found yet.")

In [ ]:
# =========================
# Make the video
# =========================

# Full run, from the first run folder to the latest:
# render_solasterid_video()

# Quicker test render, usually wise before unleashing the whole tidepool:
render_solasterid_video(max_runs=100, max_committee_scenes_per_run=100, output_path=OUTPUT_DIR / "solasterid_test_cut.mp4")

## Extra upgrades I would add next 🪸

The notebook above is already general enough to read future Solasterid anatomy, but the next delicious upgrades would be:

1. **True morphing between architectures**: interpolate old committee positions into new committee positions instead of hard-cutting at mutation scenes.
2. **Ancestry trails**: when an arm creates or influences another arm, draw glowing lineage threads.
3. **Committee traffic animation**: animate packets moving from Listenerbot to leader to members to routed committees to Speakerbot.
4. **Audio narration**: use text-to-speech for Speakerbot final findings or arm quotes, optionally one voice per role family.
5. **HTML interactive replay**: export a clickable timeline with run folders, quotes, mutation diffs, and full transcripts.
6. **Importance scoring**: choose the most meaningful quotes rather than simply the latest/first available 50 words.
7. **Architecture health HUD**: show API calls, active arm count, committee count, liveness gate state, and mutation application status as tiny dashboard glyphs.

For now, the starfish has cinema. 🎬